In [6]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "geopandas",
#     "lonboard",
#     "matplotlib",
#     "palettable",
#     "pandas",
#     "pyarrow",
#     "shapely",
# ]
# ///

import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from palettable.colorbrewer.sequential import YlGn_9
from shapely.geometry import Point

from lonboard import Map, PointCloudLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.colormap import apply_continuous_cmap
from lonboard.view_state import MapViewState

# 1. GENERATE REALISTIC 3D FOREST CANOPY LIDAR DATA
# Simulates LiDAR point cloud returns across a forest plot
np.random.seed(42)
num_trees = 150
points_per_tree = 100

# Plot geographic center (e.g., Redwood National Park)
center_lon, center_lat = -124.004, 41.213

lons, lats, heights, intensities = [], [], [], []

for _ in range(num_trees):
    # Tree trunk base position
    tree_lon = center_lon + np.random.normal(0, 0.002)
    tree_lat = center_lat + np.random.normal(0, 0.002)
    max_tree_height = np.random.uniform(15, 65)  # Tree height in meters

    # Generate 3D point cloud returns clustered around tree crown
    for _ in range(points_per_tree):
        # Height distribution biased toward upper canopy
        z = np.random.beta(2, 1) * max_tree_height
        
        # Crown radius expands with height
        crown_radius = (z / max_tree_height) * np.random.uniform(0.0001, 0.0003)
        x = tree_lon + np.random.normal(0, crown_radius)
        y = tree_lat + np.random.normal(0, crown_radius)
        
        # Intensity varies (leaves reflect higher intensity than ground)
        intensity = np.random.uniform(0.3, 1.0) if z > 2 else np.random.uniform(0.05, 0.2)

        lons.append(x)
        lats.append(y)
        heights.append(z)
        intensities.append(intensity)

# 2. CONSTRUCT 3D GEOPANDAS GEODATAFRAME
# Points constructed as 3D Shapely Geometries: Point(lon, lat, height_m)
geometry_3d = [
    Point(x, y, z)
    for x, y, z in zip(lons, lats, heights, strict=True)
]

gdf_forest = gpd.GeoDataFrame(
    {
        "canopy_height_m": heights,
        "lidar_intensity": intensities,
    },
    geometry=geometry_3d,
    crs="EPSG:4326",
)

# 3. CONTINUOUS COLOR MAPPING (Mapped to Canopy Height)
color_scale = Normalize(
    vmin=gdf_forest["canopy_height_m"].min(),
    vmax=gdf_forest["canopy_height_m"].max(),
)
fill_colors = apply_continuous_cmap(
    color_scale(gdf_forest["canopy_height_m"]), YlGn_9, alpha=0.9
)

# 4. INSTANTIATE PointCloudLayer (Lonboard API Pattern)
point_cloud_layer = PointCloudLayer.from_geopandas(
    gdf_forest,
    get_color=fill_colors,
    point_size=3.5,            # Point size in pixels
    size_units="pixels",
    pickable=True,              # Auto-populates hover tooltip with 'canopy_height_m' and 'lidar_intensity'
)

# 5. MAP VIEW WITH 3D TILT
map_ = Map(
    point_cloud_layer,
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatter),
    view_state=MapViewState(
        longitude=center_lon,
        latitude=center_lat,
        zoom=16,
        pitch=60,              # 60-degree tilt reveals vertical 3D forest canopy structure
        bearing=30,
    ),
    height=700,
    show_tooltip=True,
)

map_